In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [3]:
CSV_PATH = './data/house_prices.csv'

DEVICE = 'cpu'
if torch.cuda.is_available():
    DEVICE = 'cuda'

BATCH_SIZE = 16
EPOCHS = 1000
LEARNING_RATE = 0.01
SEED = 777

torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

In [4]:
df = pd.read_csv(CSV_PATH)
df.head(3)

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7


In [5]:
features = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
target = 'MEDV'

X = df[features]
Y = df[target]

In [6]:
# 트레이닝/테스트 분할
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=SEED
)

In [7]:
# StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train_std = scaler.transform(X_train)
X_test_std = scaler.transform(X_test)

In [8]:
# 텐서 변환
X_train_std = torch.tensor(X_train_std, dtype=torch.float32).to(DEVICE)
X_test_std = torch.tensor(X_test_std, dtype=torch.float32).to(DEVICE)
Y_train = torch.tensor(Y_train.values, dtype=torch.float32).to(DEVICE)
Y_test = torch.tensor(Y_test.values, dtype=torch.float32).to(DEVICE)

In [9]:
# Dataset / DataLoader
train_dataset = TensorDataset(X_train_std, Y_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [10]:
class Regressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(13,30),
            nn.BatchNorm1d(num_features=30),
            nn.LeakyReLU(),
            nn.Dropout(p=0.2)
        )

        self.layer2 = nn.Sequential(
            nn.Linear(30, 40),
            nn.BatchNorm1d(num_features=40),
            nn.LeakyReLU(),
            nn.Dropout(p=0.2)
        )

        self.layer3 = nn.Sequential(
            nn.Linear(40, 30),
            nn.BatchNorm1d(num_features=30),
            nn.LeakyReLU(),
            nn.Dropout(p=0.2)
        )

        self.layer4 = nn.Sequential(
            nn.Linear(30, 20),
            nn.BatchNorm1d(num_features=20),
            nn.LeakyReLU(),
            nn.Dropout(p=0.2)
        )

        self.layer5 = nn.Sequential(
            nn.Linear(20, 1)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        return x

In [12]:
model = Regressor().to(DEVICE)

In [13]:
criterion = nn.MSELoss().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [14]:
model.train()
for epoch in range(EPOCHS):
    for X_batch, Y_batch in train_loader:

        X_batch = X_batch.to(DEVICE)
        Y_batch = Y_batch.to(DEVICE)

        optimizer.zero_grad()
        Y_hat = model.forward(X_batch)
        Y_hat = Y_hat.flatten()
        loss = criterion(Y_hat, Y_batch)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')

Epoch 100, Loss: 30.0495
Epoch 200, Loss: 62.5792
Epoch 300, Loss: 2.3397
Epoch 400, Loss: 27.6976
Epoch 500, Loss: 67.9945
Epoch 600, Loss: 39.5351
Epoch 700, Loss: 30.2956
Epoch 800, Loss: 67.5729
Epoch 900, Loss: 40.3129
Epoch 1000, Loss: 19.5654


In [15]:
model.eval()
with torch.no_grad():
    Y_pred = model.forward(X_test_std)
    Y_pred = Y_pred.flatten()
    mse = torch.mean((Y_pred - Y_test)**2)
    print('MSE:', mse.item())

MSE: 12.716416358947754


In [16]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(Y_test, Y_pred)
print(mse)

12.716416


In [17]:
# MAE
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(Y_test, Y_pred)
print(mae)

2.650986


In [24]:
from sklearn.metrics import mean_squared_error
import numpy as np
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
print(rmse)

3.5660086
